# Writing Your First Operation

## What you'll learn

- How operations fit into a pipeline (the three-phase lifecycle)
- How to define a custom operation from scratch
- How to declare inputs, outputs, parameters, and lineage
- How to wire your operation into a pipeline and verify the results

**Prerequisites:** [Your First Pipeline](../01-getting-started/01-first-pipeline.ipynb)
and the source checkout from [Installation](../../getting-started/installation.md).
This example reads CSV fixtures from the checkout.
**Estimated time:** 20 minutes
**GPU required:** No.

---

## What is an operation?

An **operation** is a single unit of computation in a pipeline. In the
[first tutorial](../01-getting-started/01-first-pipeline.ipynb) you used built-in operations like
`DataGenerator` and `DataTransformer`. Now you'll build your own.

Every operation follows a three-phase lifecycle. The framework calls these
methods in order:

```
  ┌──────────────┐     ┌───────────┐     ┌───────────────┐
  │  preprocess   │ --> │  execute   │ --> │  postprocess   │
  └──────────────┘     └───────────┘     └───────────────┘
   Artifacts → paths    Paths → files     Files → artifacts
```

| Phase | What you do | What the framework provides |
|-------|-------------|---------------------------|
| **preprocess** | Extract file paths from input artifacts | `PreprocessInput` with hydrated artifact objects |
| **execute** | Run your computation, write output files | `ExecuteInput` with an output directory and the dict from preprocess |
| **postprocess** | Wrap output files as draft artifacts | `PostprocessInput` with all files from the output directory |

`preprocess` extracts the inputs your computation needs. `execute_function`
reads those files and writes results. `postprocess` returns those results as
artifacts so the pipeline can store them and record their lineage.

---

## What we'll build

A **TextUppercaser** operation that:
1. Takes text files as input
2. Converts their contents to uppercase
3. Optionally prepends a header line
4. Returns the results as new data artifacts with provenance tracked



---

## Imports

These are the building blocks for any operation.

In [ ]:
from __future__ import annotations

from enum import StrEnum
from typing import Any, ClassVar

from pydantic import BaseModel, Field

from artisan.operations.base import OperationDefinition, PerArtifact
from artisan.orchestration import StepStatus
from artisan.schemas import (
    ArtifactResult,
    DataArtifact,
    ExecuteInput,
    InputSpec,
    OutputSpec,
    PostprocessInput,
    PreprocessInput,
)

---

## Define the operation skeleton

Every operation needs four things declared at the class level:

1. **Metadata** — `name` and `description` (how the framework identifies your operation)
2. **Role enums** — `InputRole` and `OutputRole` (named slots for data flowing in and out)
3. **Specs** — `inputs` and `outputs` dicts mapping roles to `InputSpec`/`OutputSpec`
4. **Params** — a nested Pydantic model for algorithm-specific configuration (optional)

Let's define all four, then implement the lifecycle methods one at a time.

The document role is used on both sides. Declare its type and lineage:

```python
inputs = {"document": InputSpec(artifact_type="data", required=True)}
outputs = {
    "document": OutputSpec(
        artifact_type="data", infer_lineage_from={"inputs": ["document"]}
    )
}
```

The complete class below includes these declarations and the matching role enums.

A few things to notice:

- **`InputRole` and `OutputRole`** are `StrEnum`s whose values must exactly match
  the keys in `inputs` and `outputs`. The framework validates this at class
  creation time.
- **`infer_lineage_from={"inputs": ["document"]}`** tells the framework that each
  output artifact descends from an input in the `"document"` role. This is how
  [provenance](../../concepts/provenance-system.md) edges are created
  automatically — you declare the relationship, the framework records it.
- **`Params`** is a Pydantic model that validates the provided values. Pipeline users pass params as a dict; the framework constructs the model.

---

## Implement preprocess

`preprocess` receives a `PreprocessInput` containing hydrated artifact objects.
Your job: extract what `execute` needs (usually file paths) and return a plain
dict.

This is the most common preprocess pattern — turn artifacts into paths:

```python
def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
    return {
        role: PerArtifact([artifact.materialized_path for artifact in artifacts])
        for role, artifacts in inputs.input_artifacts.items()
    }

```


`materialized_path` points to a local file prepared for execution. Wrap the
per-artifact path list in `PerArtifact` so each execute call receives its own
input. A plain list is shared data, such as common settings, and is passed
unchanged to every call in the unit.

---

## Implement execute

`execute_function` performs the computation. It receives:

- `inputs.inputs` — the dict returned by `preprocess`
- `inputs.execute_dir` — a directory where you write output files

Write your results to `execute_dir`. The framework collects all files in that
directory and passes them to `postprocess`.

```python
def execute_function(self, inputs: ExecuteInput) -> Any:
    import os

    output_dir = inputs.execute_dir
    os.makedirs(output_dir, exist_ok=True)

    for input_path in inputs.inputs["document"]:
        text = open(input_path).read()
        result = text.upper()
        if self.params.add_header:
            result = f"=== UPPERCASED ===\n{result}"
        open(os.path.join(output_dir, os.path.basename(input_path)), "w").write(result)

    return {"processed": True}

```


Notice that `execute` accesses `self.params.add_header` — parameters are
available on the instance. The return value (here `{"processed": True}`) is
optional; `postprocess` can access it via `inputs.memory_outputs`, but for
file-based operations the output files are usually sufficient.

---

## Implement postprocess

`postprocess` converts the files `execute` wrote into **draft artifacts** that
the framework can commit to Delta Lake. It receives:

- `inputs.file_outputs` — list of all files in `execute_dir`
- `inputs.step_number` — needed when creating draft artifacts
- `inputs.memory_outputs` — whatever `execute` returned

```python
from pathlib import Path

def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
    drafts = [
        DataArtifact.draft(
            content=Path(f).read_bytes(),
            original_name=Path(f).name,
            step_number=inputs.step_number,
        )
        for f in inputs.file_outputs
    ]
    return ArtifactResult(success=True, artifacts={"document": drafts})

```


`DataArtifact.draft()` creates a **draft** artifact — it has no `artifact_id`
yet. Before commit, the framework finalizes it by computing a typed
[content-addressed](../../concepts/design-principles.md) ID from its canonical
payload and semantic identity metadata.

The dict key `"document"` in `ArtifactResult.artifacts` must match your
`OutputRole` value.

---

## Assemble the complete operation

Now let's put all three lifecycle methods together into the real class.
This is the complete, working operation.

In [ ]:
from pathlib import Path


class TextUppercaser(OperationDefinition):
    """Uppercase the contents of text files."""

    name = "text_uppercaser"
    description = "Convert text file contents to uppercase"

    class InputRole(StrEnum):
        DOCUMENT = "document"

    class OutputRole(StrEnum):
        DOCUMENT = "document"

    inputs: ClassVar[dict[str, InputSpec]] = {
        InputRole.DOCUMENT: InputSpec(artifact_type="data", required=True),
    }
    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.DOCUMENT: OutputSpec(
            artifact_type="data",
            infer_lineage_from={"inputs": ["document"]},
        ),
    }

    class Params(BaseModel):
        add_header: bool = Field(default=False, description="Prepend a header line")

    params: Params = Params()

    def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
        return {
            role: PerArtifact([artifact.materialized_path for artifact in artifacts])
            for role, artifacts in inputs.input_artifacts.items()
        }

    def execute_function(self, inputs: ExecuteInput) -> Any:
        import os

        output_dir = inputs.execute_dir
        os.makedirs(output_dir, exist_ok=True)

        for input_path in inputs.inputs["document"]:
            text = open(input_path).read()
            result = text.upper()
            if self.params.add_header:
                result = f"=== UPPERCASED ===\n{result}"
            open(os.path.join(output_dir, os.path.basename(input_path)), "w").write(
                result
            )

        return {"processed": True}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        drafts = [
            DataArtifact.draft(
                content=Path(f).read_bytes(),
                original_name=Path(f).name,
                step_number=inputs.step_number,
            )
            for f in inputs.file_outputs
        ]
        return ArtifactResult(success=True, artifacts={"document": drafts})


print(f"Operation '{TextUppercaser.name}' ready")
print(f"  Inputs:  {list(TextUppercaser.inputs.keys())}")
print(f"  Outputs: {list(TextUppercaser.outputs.keys())}")
print(f"  Params:  {TextUppercaser.Params.model_fields.keys()}")

---

## Wire it into a pipeline

Use `IngestData` to bring text files into the
[provenance graph](../../concepts/provenance-system.md), then pass them to
our custom operation.

```
Step 0: IngestData          Step 1: TextUppercaser
┌──────────────────┐       ┌──────────────────────┐
│ Text files on disk │ --->  │ Uppercase + header    │
└──────────────────┘       └──────────────────────┘
   outputs: "data"            inputs: "document"
                              outputs: "document"
```

In [ ]:
from artisan.operations.curator import IngestData
from artisan.orchestration import PipelineManager
from artisan.utils import find_project_root, tutorial_setup

# Find some CSV files to use as input
PROJECT_ROOT = find_project_root()
SOURCE_FILES = sorted((PROJECT_ROOT / "tests" / "fixtures" / "csv").glob("*.csv"))[:2]

print(f"Source files: {[f.name for f in SOURCE_FILES]}")

In [ ]:
from fsspec.implementations.local import LocalFileSystem

from artisan.storage import ArtifactStore

expected = {path.stem: path.read_text().upper().encode() for path in SOURCE_FILES}
assert len(expected) == 2

for artifacts_per_unit in (1, 2):
    env = tutorial_setup(f"writing_an_operation_{artifacts_per_unit}")
    pipeline = PipelineManager.create(
        name="uppercaser_demo",
        delta_root=env.delta_root,
        staging_root=env.staging_root,
        working_root=env.working_root,
    )
    output = pipeline.output
    pipeline.run(IngestData, name="ingest", inputs=[str(f) for f in SOURCE_FILES])
    step1 = pipeline.run(
        TextUppercaser,
        name="uppercase",
        inputs={"document": output("ingest", "data")},
        batch_strategy={"artifacts_per_unit": artifacts_per_unit},
    )
    summary = pipeline.finalize()
    assert step1.status is StepStatus.SUCCEEDED
    store = ArtifactStore(env.delta_root, fs=LocalFileSystem())
    ids = store.provenance.load_artifact_ids_by_type("data", step_numbers=[1])
    actual = store.get_artifacts_by_type(list(ids), "data")
    assert len(actual) == 2
    assert {
        artifact.original_name: artifact.content for artifact in actual.values()
    } == expected
    print(
        f"Verified both uppercase files with {artifacts_per_unit} artifact(s) per unit"
    )

The two runs check the same output bytes with one and two artifacts per unit.
`PerArtifact` keeps each execute call tied to the correct document in both cases.
The inspections below show the second run.

`inputs={"document": output("ingest", "data")}` connects the ingested data to the
operation’s document role. We leave `add_header=False` so the outputs remain
CSV files that `inspect_data` can display.

---

## Inspect the results

Let's verify that both steps ran and our custom operation produced the
expected artifacts.

In [ ]:
from artisan.visualization import inspect_pipeline

inspect_pipeline(env.delta_root)

The table should show two steps: `ingest` (step 0) and
`uppercase` (step 1), both with status `succeeded`. The `produced` column
confirms that our operation created data artifacts.

In [ ]:
from artisan.visualization import inspect_data

# Compare input data (step 0) with output data (step 1)
print("=== Input data (step 0) ===")
display(inspect_data(env.delta_root, step_number=0))

print("\n=== Output data (step 1 — uppercased) ===")
display(inspect_data(env.delta_root, step_number=1))

### Provenance

The macro graph shows data flow from IngestData to TextUppercaser.
Because we declared `infer_lineage_from={"inputs": ["document"]}`,
each output artifact has a provenance edge back to its input — no
extra code required.

In [ ]:
from artisan.visualization import build_macro_graph

build_macro_graph(env.delta_root)

In [ ]:
print(f"Pipeline '{summary['pipeline_name']}' complete")
print(f"  Steps: {summary['total_steps']}")
print(f"  Success: {summary['overall_success']}")

## Reuse the operation

The complete `TextUppercaser` class above can move into an importable module.
Keep the input/output declarations next to the computation so its file contract
and lineage remain clear. [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md)
covers additional patterns; the [Python API guide](../../reference/python-api.md)
points to current method and model docstrings.

---

## Summary

You built a custom operation from scratch:

1. **Declared** metadata, roles, specs, and params
2. **Implemented** the three-phase lifecycle (`preprocess` / `execute` / `postprocess`)
3. **Set lineage** with `infer_lineage_from` so provenance is tracked automatically
4. **Wired** the operation into a pipeline with `output("step_name", "role")`
5. **Verified** the results with `inspect_pipeline`, `inspect_data`, and provenance graphs

## Next steps

- [Operations Model](../../concepts/operations-model.md) — Deeper understanding of how operations work and why they're designed this way
- [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md) — Task-oriented recipes for advanced patterns (generative ops, multi-input, external tools)
- [Writing Curator Operations](../../how-to-guides/writing-curator-operations.md) — Filter, Merge, and Ingest patterns
- [Glossary](../../reference/glossary.md) — Key terms and definitions
- [Pipeline Patterns](../02-pipeline-design/01-sources-and-sequencing.ipynb) — Reusable pipeline topologies